# Digital Image Processing Project
## Fruit Sorting System by Color and Size

**Objective**: Create an industrial pipeline to sort fruits using advanced image processing techniques.

---

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from google.colab import drive

# Constants for tuning
GAUSSIAN_KERNEL = (5, 5)
MEDIAN_KSIZE = 5
FIGURE_SIZE = (12, 6)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 1: Pre-processing
การปรับปรุงคุณภาพภาพเพื่อลด Noise และเพิ่ม Contrast ให้วัตถุแยกออกจากพื้นหลังได้ชัดเจนที่สุด

### 1.1 Helper Functions (Visualization)
สร้างฟังก์ชันสำหรับการแสดงผลภาพเปรียบเทียบแบบ Side-by-Side

In [2]:
def show_comparison(img1: np.ndarray, img2: np.ndarray, title1: str = "Original", title2: str = "Processed") -> None:
    """Displays two images side by side for easy comparison."""
    plt.figure(figsize=FIGURE_SIZE)

    # Left image
    plt.subplot(1, 2, 1)
    if len(img1.shape) == 3:
        plt.imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img1, cmap='gray')
    plt.title(title1)
    plt.axis('off')

    # Right image
    plt.subplot(1, 2, 2)
    if len(img2.shape) == 3:
        plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img2, cmap='gray')
    plt.title(title2)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

### 1.2 Image Acquisition

In [4]:
def load_image(path: str) -> Optional[np.ndarray]:
    """Loads an image with basic error handling."""
    try:
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Cannot find image at {path}")
        return img
    except Exception as e:
        print(f"Error: {e}")
        return None

### 1.3 Histogram Equalization (Manual Implementation)
การใช้ CDF เพื่อกระจายความเข้มแสงให้ทั่วภาพ ช่วยให้รายละเอียดในส่วนที่มืดหรือสว่างเกินไปชัดเจนขึ้น

In [5]:
def manual_histogram_equalization(gray_img: np.ndarray) -> np.ndarray:
    """Manually implements global histogram equalization."""
    # Ensure image is grayscale
    if len(gray_img.shape) != 2:
        gray_img = cv2.cvtColor(gray_img, cv2.COLOR_BGR2GRAY)

    # 1. Histogram Calculation
    hist, _ = np.histogram(gray_img.flatten(), 256, [0, 256])

    # 2. CDF Calculation
    cdf = hist.cumsum()

    # 3. CDF Normalization (Formula based on DIP Theory)
    cdf_m = np.ma.masked_equal(cdf, 0)
    cdf_m = (cdf_m - cdf_m.min()) * 255 / (cdf_m.max() - cdf_m.min())
    equalized_lookup_table = np.ma.filled(cdf_m, 0).astype('uint8')

    # 4. Image Mapping
    return equalized_lookup_table[gray_img]

### 1.4 Noise Reduction Filters

In [6]:
def denoise_image(img: np.ndarray, method: str = 'gaussian') -> np.ndarray:
    """
    Reduces noise using specified method.
    'gaussian': Smooths general electronic noise.
    'median': Best for salt-and-pepper noise.
    """
    if method.lower() == 'gaussian':
        return cv2.GaussianBlur(img, GAUSSIAN_KERNEL, 0)
    elif method.lower() == 'median':
        return cv2.medianBlur(img, MEDIAN_KSIZE)
    else:
        return img

In [ ]:
# ใส่ตำแหน่งที่อยู่ของไฟล์ภาพใน Google Drive ของคุณ
image_path = ''

# 1. ทดสอบการโหลดภาพ
original_img = load_image(image_path)

if original_img is not None:
    # 2. ทดลองลด Noise ทำ Gaussian Blur
    denoised_img = denoise_image(original_img, method='gaussian')
    show_comparison(original_img, denoised_img, "Original Image", "After Gaussian Blur")

    # 3. ทดลองปรับความสว่าง (Histogram Equalization)
    gray_img = cv2.cvtColor(denoised_img, cv2.COLOR_BGR2GRAY)
    eq_img = manual_histogram_equalization(gray_img)
    show_comparison(gray_img, eq_img, "Grayscale Image", "Manual Histogram Equalization")


## Section 2: Step 2 — Color Segmentation
การแปลง Color Space (BGR ไปเป็น HSV) เพื่อให้ง่ายต่อการตรวจจับสีผลไม้ และใช้ Mask หรือเทคนิค Otsu เพื่อตีกรอบวัตถุ

In [ ]:
def create_color_mask(bgr_img: np.ndarray, color_type: str = 'red') -> np.ndarray:
    """Converts BGR to HSV and returns a binary mask for the specified color."""
    hsv_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    
    if color_type.lower() == 'red':
        mask1 = cv2.inRange(hsv_img, np.array([0, 100, 100]), np.array([10, 255, 255]))
        mask2 = cv2.inRange(hsv_img, np.array([170, 100, 100]), np.array([180, 255, 255]))
        return cv2.bitwise_or(mask1, mask2)
        
    elif color_type.lower() == 'yellow':
        return cv2.inRange(hsv_img, np.array([15, 100, 100]), np.array([35, 255, 255]))
        
    elif color_type.lower() == 'green':
        return cv2.inRange(hsv_img, np.array([36, 50, 50]), np.array([85, 255, 255]))
        
    return np.zeros(hsv_img.shape[:2], dtype=np.uint8)

In [ ]:
def otsu_segmentation(bgr_img: np.ndarray) -> np.ndarray:
    """Uses Otsu's thresholding to segment object from background."""
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_img, GAUSSIAN_KERNEL, 0)
    _, binary_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Invert mask if background is white 
    inverted_mask = cv2.bitwise_not(binary_mask)
    return inverted_mask

In [ ]:
if original_img is not None:
    print("--- Testing Segmentation Algorithms ---")
    
    otsu_mask = otsu_segmentation(original_img)
    show_comparison(original_img, otsu_mask, "Original Image", "Otsu's Binarization")
    
    mask_red = create_color_mask(original_img, 'red')
    mask_yellow = create_color_mask(original_img, 'yellow')
    mask_green = create_color_mask(original_img, 'green')
    
    show_comparison(original_img, mask_green, "Original", "Green Mask (e.g. Grapes)")
    show_comparison(mask_red, mask_yellow, "Red Mask", "Yellow Mask (e.g. Mango/Banana)")


## Section 3: Step 3 — Morphological Operations & Contour Detection
การทำความสะอาดจุดน๊อยส์บน Mask และการค้นหาเส้นขอบ (Contour) เพื่อคำนวณขนาดวัตถุ

### 3.1 Morphological Operations
ใช้ Erosion และ Dilation (ผ่าน Opening/Closing) ทำให้กรอบผลไม้เนียนขึ้น ไม่มีรอยแหว่ง

In [ ]:
def apply_morphology(binary_mask: np.ndarray) -> np.ndarray:
    """Cleans up the binary mask using Opening (remove noise) and Closing (fill holes)."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)
    cleaned_mask = cv2.morphologyEx(cleaned_mask, cv2.MORPH_CLOSE, kernel)
    
    return cleaned_mask

### 3.2 Contour Detection
ใช้ `cv2.findContours()` เพื่อวาดเส้นล้อมรอบกรอบขาวบน Mask และกรองเอาเฉพาะวัตถุที่ใหญ่กว่า min_area ที่ตั้งไว้

In [ ]:
def find_fruit_contours(cleaned_mask: np.ndarray, original_img: np.ndarray, min_area: int = 500) -> Tuple[np.ndarray, list]:
    """Finds contours from the mask and filters out small noise contours based on area."""
    # ค้นหาเส้นขอบ
    contours, hierarchy = cv2.findContours(cleaned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]
    output_img = original_img.copy()
    cv2.drawContours(output_img, valid_contours, -1, (0, 255, 0), 2)
    return output_img, valid_contours

### 3.3 Test Morphology & Contours
ทดสอบทำความสะอาด Mask และวาดเส้นตีกรอบผลองุ่น/มะม่วง

In [ ]:
if original_img is not None:
    print("--- Testing Morphology and Contours ---")
    cleaned_mask = apply_morphology(otsu_mask)
    show_comparison(otsu_mask, cleaned_mask, "Mask Before (Otsu)", "Mask After Morphology")
    contour_img, fruit_contours = find_fruit_contours(cleaned_mask, original_img)
    show_comparison(original_img, contour_img, "Original Image", f"Found {len(fruit_contours)} Fruit(s)!")


## Section 4: Step 4 — Feature Extraction
ขั้นตอนการเปลี่ยนรูปทรงภาพเป็น "ข้อมูลตัวเลข (Features)" เพื่อเอาไปสร้างเงื่อนไขจัดเกรดผลไม้

### 4.1 Extract Features Function
คำนวณลักษณะเด่น (พื้นที่, ความกลม, อัตราส่วน, และสีเฉลี่ย) จาก Contour แต่ละอันที่เราหาเจอในรูปภาพ

In [ ]:
def extract_features(contour: np.ndarray, bgr_img: np.ndarray) -> dict:
    """Extracts mathematical features from a contour such as area, circularity, and mean color."""
    # แปลงเป็น HSV เพื่อดึงค่าสี
    hsv_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    
    # 1. ขนาด (Area) และเส้นรอบรูป (Perimeter)
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    
    # 2. ความกลม (Circularity) // ยิ่งใกล้ 1.0 แปลว่ายิ่งกลม
    circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
    
    # 3. กรอบสี่เหลี่ยม (Bounding Box) และอัตราส่วน (Aspect Ratio)
    x, y, w, h = cv2.boundingRect(contour)
    aspect_ratio = w / h if h > 0 else 0
    
    # 4. สีเฉลี่ย (Average Color) ในกรอบผลไม้นั้นๆ
    mask = np.zeros(hsv_img.shape[:2], dtype=np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1) # วาด Mask สีขาวเฉพาะเนื้อผลไม้
    mean_color = cv2.mean(hsv_img, mask=mask)[:3]  # หาค่าเฉลี่ย HSV (Hue, Saturation, Value)
    
    return {
        'area': area,
        'perimeter': perimeter,
        'circularity': circularity,
        'aspect_ratio': aspect_ratio,
        'mean_hue': mean_color[0],
        'mean_saturation': mean_color[1],
        'mean_value': mean_color[2],
        'bounding_box': (x, y, w, h)
    }

### 4.2 Test Feature Extraction
ทดสอบสกัดข้อมูลตัวเลขจากผลไม้ที่เราตีกรอบไว้ใน Step ที่แล้ว

In [ ]:
if original_img is not None and len(fruit_contours) > 0:
    print("--- Extracted Features for Found Fruits ---")
    
    # ทดสอบดึง Feature ผลไม้ทุกลูกที่จับได้
    for i, contour in enumerate(fruit_contours):
        features = extract_features(contour, original_img)
        print(f"\nFruit #{i+1}:")
        print(f"  - Area (ขนาด): {features['area']:.2f} pixels")
        print(f"  - Circularity (ความกลม): {features['circularity']:.2f}")
        print(f"  - Aspect Ratio (สัดส่วน): {features['aspect_ratio']:.2f}")
        print(f"  - Average Hue (เฉดสี): {features['mean_hue']:.2f}")


## Section 5: Step 5 — Classification & Results
การใช้กฎ (Rule-based) เพื่อตัดสินว่าวัตถุที่ตรวจจับได้คือผลไม้ชนิดไหน และขนาดเท่าไหร่

In [ ]:
def classify_fruit(features: dict) -> dict:
    """Classifies fruit based on hue and size (area)."""
    hue = features['mean_hue']
    area = features['area']
    
    # 1. จำแนกสี (Color Class)
    if (0 <= hue <= 12) or (165 <= hue <= 180):
        color_label = "Red (Strawberry/Apple)"
    elif 13 <= hue <= 25:
        color_label = "Orange (Orange/Mango)"
    elif 26 <= hue <= 38:
        color_label = "Yellow (Banana/Mango)"
    elif 39 <= hue <= 90:
        color_label = "Green (Green Grape/Mango)"
    else:
        color_label = "Unknown"

    # 2. จำแนกขนาด (Size Class) - ปรับ Threshold ตามข้อมูลตัวอย่าง
    if area < 5000:
        size_label = "Small"
    elif area < 15000:
        size_label = "Medium"
    else:
        size_label = "Large"
        
    return {"color": color_label, "size": size_label}

In [ ]:
if original_img is not None and len(fruit_contours) > 0:
    print("--- Final Classification Results ---")
    result_img = original_img.copy()
    
    for i, contour in enumerate(fruit_contours):
        features = extract_features(contour, original_img)
        prediction = classify_fruit(features)
        
        # แสดงผลลัพธ์ทางหน้าจอ
        print(f"Fruit #{i+1}: {prediction['color']} | Size: {prediction['size']}")
        
        # วาดข้อความทับลงบรูปภาพ
        x, y, w, h = features['bounding_box']
        text = f"#{i+1}: {prediction['size']}"
        cv2.putText(result_img, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.rectangle(result_img, (x, y), (x+w, y+h), (0, 255, 0), 2)

    show_comparison(original_img, result_img, "Original", "Classified Output")

## Section 6: System Evaluation (Batch Processing)
ทดสอบความแม่นยำของระบบคัดแยกผลไม้โดยรันภาพแบบอัตโนมัติหลายสิบรูป เพื่อทำ Report ส่งอาจารย์ (เป้าหมาย Accuracy > 95%)

In [ ]:
import os
import glob

def evaluate_system_accuracy(base_folder_path: str, max_images_per_class: int = 50):
    """
    โหลดภาพจากโฟลเดอร์ผลไม้แต่ละชนิด นำมาวิ่งผ่าน Pipeline ทั้งหมดตั้งแต่ต้นยันจบ
    เพื่อคำนวณหา % ความแม่นยำของระบบ (Accuracy Rate)
    """
    total_images = 0
    correct_predictions = 0
    
    # เตรียมชื่อโฟลเดอร์
    valid_classes = ['Grape', 'Mango', 'Apple', 'Banana', 'Strawberry']
    
    print(f"{'-Fruit Class-':<15} | {'-Tested-':<8} | {'-Correct-':<8} | {'-Accuracy %-':<10}")
    print("-" * 50)
    
    for fruit_name in valid_classes:
        folder_path = os.path.join(base_folder_path, fruit_name)
        if not os.path.exists(folder_path):
            continue
            
        # ดึงไฟล์ .jpeg, .jpg, .png ออกมาทั้งหมด
        images = glob.glob(os.path.join(folder_path, "*.*"))
        images = images[:max_images_per_class] # จำกัดจำนวนรูปไม่ให้รันนานเกินไป
        
        class_total = 0
        class_correct = 0
        
        for img_path in images:
            img = load_image(img_path)
            if img is None: continue
            
            # --- วิ่งผ่าน PIPELINE อัตโนมัติ ---
            denoised_img = denoise_image(img, method='gaussian')
            otsu_mask = otsu_segmentation(denoised_img)
            cleaned_mask = apply_morphology(otsu_mask)
            _, contours = find_fruit_contours(cleaned_mask, denoised_img, min_area=500)
            
            if not contours:
                class_total += 1
                continue
                
            # เลือกผลไม้ชิ้นที่ใหญ่ที่สุดในภาพมาวิเคราะห์
            largest_contour = max(contours, key=cv2.contourArea)
            features = extract_features(largest_contour, denoised_img)
            prediction = classify_fruit(features)
            # ---------------------------
            
            # ตรวจสอบว่าระบบทายถูกไหม (ดูว่ามีคำว่า Grape/Mango ซ่อนใน Output สีไหม)
            if fruit_name in prediction['color']:
                class_correct += 1
            
            class_total += 1
            
        if class_total > 0:
            accuracy = (class_correct / class_total) * 100
            print(f"{fruit_name:<15} | {class_total:<8} | {class_correct:<8} | {accuracy:>8.2f}%")
            total_images += class_total
            correct_predictions += class_correct
            
    # พิมพ์สรุปผลรวมด้านล่างสุด
    if total_images > 0:
        overall_accuracy = (correct_predictions / total_images) * 100
        print("-" * 50)
        print(f"OVERALL ACCURACY: {overall_accuracy:.2f}% ({correct_predictions}/{total_images} images)")
        if overall_accuracy >= 95.0:
            print("PASSED: ความแม่นยำ >95%")
        else:
            print("NEEDS TUNING: ลองกลับไปปรับค่าตัวเลข Threshold อีกครั้ง")

In [ ]:
dataset_path = "/content/drive/MyDrive/Colab Notebooks/CS483/Project/data"
evaluate_system_accuracy(dataset_path, 50)